# 14 RIB-Assisted Normal Verification

This notebook re-evaluates the existing six TRAIN-only Normal candidate periods using RouteViews `route-views2` RIB snapshots plus UPDATE activity. It does not select new Normal candidates, alter frozen v2 data, use Validation/Holdout, train models, or add CAIDA/RPKI features.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

periods = pd.read_csv(PILOT / 'normal_mrt_pilot_periods_v1.csv')
rib_plan = pd.read_csv(PILOT / 'normal_rib_acquisition_plan_v1.csv')
rib_download = pd.read_csv(PILOT / 'normal_rib_download_audit_v1.csv')
rib_stability = pd.read_csv(PILOT / 'normal_rib_stability_v1.csv')
update_only = pd.read_csv(PILOT / 'normal_pilot_qualification_results_v1.csv')
rib_update = pd.read_csv(PILOT / 'normal_rib_update_qualification_results_v1.csv')
sensitivity = pd.read_csv(PILOT / 'normal_rib_sustained_stability_sensitivity_v1.csv')
method_comparison = pd.read_csv(PILOT / 'normal_method_comparison_v1.csv')

## Scope Checks

In [ ]:
assert len(periods) == 6
assert set(periods['partition']) == {'train'}
assert periods['associated_category'].value_counts().to_dict() == {'Hijack': 2, 'Leak': 2, 'Outage': 2}
periods

## RIB Acquisition Plan

The RouteViews archive was inspected to choose the nearest available before/after RIB snapshots for each candidate period. The discovered cadence is recorded in the plan rather than assumed.

In [ ]:
{
    'candidate_periods': len(rib_plan),
    'all_periods_remote_available': bool((rib_plan['remote_available'] == True).all()),
    'unique_rib_files': len(set(rib_plan['rib_before_url']).union(set(rib_plan['rib_after_url']))),
    'estimated_mib': round((rib_plan['rib_before_estimated_content_length'].sum() + rib_plan['rib_after_estimated_content_length'].sum()) / 1024 / 1024, 2),
}

In [ ]:
rib_plan[['candidate_period_id', 'rib_before_timestamp', 'rib_after_timestamp', 'remote_available', 'archive_cadence_notes']]

## Download Integrity

In [ ]:
assert len(rib_download) == 12
assert (rib_download['downloaded'] == True).all()
assert (rib_download['compression_valid'] == True).all()
rib_download[['mrt_url', 'file_size_bytes', 'sha256', 'compression_valid']]

## RIB Stability Evidence

The pilot uses bounded RIB evidence scans. Exact prefixes are inspected directly when available; ASN-only periods use ASN/path-origin evidence. This is adequate for the pilot verdict but should be replaced with indexed or compiled full-RIB processing before full TRAIN scale-up.

In [ ]:
visibility = rib_stability.groupby('candidate_period_id').agg(
    route_presence_stable=('route_presence_stable', lambda x: sorted(set(map(str, x)))),
    origin_stable=('origin_stable', lambda x: sorted(set(map(str, x)))),
    timed_out_before=('rib_scan_timed_out_before', lambda x: sorted(set(map(str, x)))),
    timed_out_after=('rib_scan_timed_out_after', lambda x: sorted(set(map(str, x)))),
    matched_prefix_rows=('prefix', 'count'),
)
visibility

## Method Comparison

In [ ]:
method_comparison

In [ ]:
rib_update['final_state'].value_counts()

In [ ]:
rib_update.merge(
    periods[['candidate_period_id', 'associated_category', 'associated_incident_uid', 'candidate_offset']],
    on='candidate_period_id',
).groupby(['associated_category', 'associated_incident_uid', 'candidate_offset'])['final_state'].value_counts().unstack(fill_value=0)

## Sustained Stability Sensitivity

In [ ]:
sensitivity

## Pilot Verdict

The methodological answer is yes: RIB plus UPDATE evidence can produce trustworthy Normal windows where route visibility and origin stability are confirmed. The UPDATE-only method falsely rejected stable silence because UPDATE archives record changes, not persistent state. For this pilot, the 30-minute provisional rule verifies 53 of 63 windows. The process is not ready to scale exactly as implemented because full route-views2 RIB inspection needs a faster indexed or compiled workflow, and one candidate period was not RIB-confirmable.